# Lightweight Machine Learning Model for Early Diabetes Prediction

## IT41033 - Nature Inspired Algorithms

### Dataset
PIMA Indians Diabetes Dataset

### Models
1. Logistic Regression
2. Decision Tree
3. Gaussian Naive Bayes
4. Random Forest

## Problem Statement

Diabetes is often diagnosed only after serious health complications have
developed. Machine-learning models can analyse routine clinical measurements
and support early diabetes screening.

This project aims to develop and compare lightweight classification models
using the PIMA Indians Diabetes Dataset. The selected models will be evaluated
based on their predictive performance, reliability, model size and inference
speed.

## Research Questions

1. Which clinical attributes provide the strongest predictive power for
   diabetes prediction?

2. Can lightweight classifiers achieve reliable diabetes-prediction
   performance?

3. Which model provides the best balance between accuracy, recall, model size
   and prediction speed?

## Objectives

1. Collect and understand the PIMA Indians Diabetes Dataset.
2. Identify missing, invalid, duplicate and unusual values.
3. Clean and normalize the clinical attributes.
4. Select the most useful features using correlation analysis and RFE.
5. Train Logistic Regression, Decision Tree and Gaussian Naive Bayes models.
6. Train a Random Forest model as the comparison baseline.
7. Evaluate the models using accuracy, precision, recall, F1-score and ROC-AUC.
8. Apply cross-validation and hyperparameter tuning.
9. Select the best-performing lightweight model.
10. Measure the selected model's size and inference time.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv("/content/diabetes.csv")

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully
Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


# Step 3: Data Preprocessing

This section prepares the dataset for model training. Invalid zero measurements
will be converted into missing values, followed by train-test splitting,
median imputation, normalization and feature selection.

The original dataframe will be preserved, and preprocessing will be performed
using a separate working copy.

In [3]:
# Create a separate copy for preprocessing
df_clean = df.copy()

# Columns where zero represents an invalid measurement
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

# Convert invalid zero values into NaN
df_clean[invalid_zero_columns] = (
    df_clean[invalid_zero_columns].replace(0, np.nan)
)

# Check the new missing-value counts
print("Missing values after converting invalid zeros to NaN:")
print(df_clean.isnull().sum())

print("\nTotal missing values:", df_clean.isnull().sum().sum())

Missing values after converting invalid zeros to NaN:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

Total missing values: 652


All 652 invalid zero measurements are now represented as NaN, while the original dataframe remains unchanged.

We must split the data before median imputation, scaling and feature selection. This prevents information from the test set leaking into the training process.

## 3.2 Feature-Target Separation and Train-Test Split

The eight clinical attributes are separated from the Outcome target. The data
is divided into 80% training data and 20% testing data.

Stratified splitting is used to preserve the original proportion of diabetic and non-diabetic patients in both sets.

In [4]:
# Separate clinical features and target
X = df_clean.drop(columns=["Outcome"])
y = df_clean["Outcome"]

# Create stratified 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Full feature dataset:", X.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining Outcome percentages:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTesting Outcome percentages:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

Full feature dataset: (768, 8)
Training features: (614, 8)
Testing features: (154, 8)
Training target: (614,)
Testing target: (154,)

Training Outcome percentages:
Outcome
0    65.15
1    34.85
Name: proportion, dtype: float64

Testing Outcome percentages:
Outcome
0    64.94
1    35.06
Name: proportion, dtype: float64


The split is properly stratified:

Training set: 614 patients
Testing set: 154 patients
Both sets maintain approximately 65% non-diabetic and 35% diabetic records

## 3.3 Median Imputation

Invalid measurements represented as NaN are replaced using median values.

The imputer is fitted only on the training data to prevent test-data
information from influencing model training. The learned training medians are
then applied to both training and testing datasets.

In [5]:
# Create the median imputer
imputer = SimpleImputer(strategy="median")

# Learn medians from training data and transform both sets
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Convert the results back into dataframes
X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)

# Display the median used for each feature
imputation_values = pd.Series(
    imputer.statistics_,
    index=X_train.columns,
    name="Training Median"
)

print("Median values learned from the training data:")
print(imputation_values)

print("\nMissing values remaining in training data:")
print(X_train_imputed.isnull().sum().sum())

print("Missing values remaining in testing data:")
print(X_test_imputed.isnull().sum().sum())

Median values learned from the training data:
Pregnancies                   3.0000
Glucose                     117.0000
BloodPressure                72.0000
SkinThickness                29.0000
Insulin                     125.0000
BMI                          32.4000
DiabetesPedigreeFunction      0.3825
Age                          29.0000
Name: Training Median, dtype: float64

Missing values remaining in training data:
0
Missing values remaining in testing data:
0


Median imputation worked successfully:

Training missing values: 0

Testing missing values: 0

Imputation values were learned only from the training data

## 3.4 Feature Normalization

The clinical features have different measurement units and numerical ranges.
StandardScaler is used to transform each feature to approximately zero mean and
unit variance.

The scaler is fitted only on the imputed training dataset. The same fitted
scaler is then applied to the testing dataset to prevent data leakage.

In [9]:
# Create

Median imputation worked properly:

The imputer learned values only from the training data.
Training missing values remaining: 0
Testing missing values remaining: 0

## 3.4 Feature Standardization

The clinical features use different numeric scales. StandardScaler transforms
each feature so that the training data has approximately zero mean and unit
standard deviation.

The scaler is fitted only on the imputed training data. The same learned
transformation is then applied to the testing data to prevent data leakage.

In [10]:
# Create the feature scaler
scaler = StandardScaler()

# Fit on training data and transform both sets
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert scaled arrays back into dataframes
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train_imputed.columns,
    index=X_train_imputed.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test_imputed.columns,
    index=X_test_imputed.index
)

print("First five scaled training records:")
display(X_train_scaled.head())

print("\nAverage of each scaled training feature:")
print(X_train_scaled.mean().round(4))

print("\nStandard deviation of each scaled training feature:")
print(X_train_scaled.std(ddof=0).round(4))

First five scaled training records:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
353,-0.851355,-1.056427,-0.826740,-1.918187,-1.203361,-0.769694,0.310794,-0.792169
711,0.356576,0.144399,0.477772,-0.229874,-1.470195,-0.417714,-0.116439,0.561034
373,-0.549372,-0.556083,-1.152868,1.233330,-0.555335,0.359576,-0.764862,-0.707594
46,-0.851355,0.811525,-1.315932,-0.004766,-0.161437,-0.403048,0.262314,-0.369293
682,-1.153338,-0.889646,-0.663676,1.120776,-0.415565,1.782163,-0.337630,-0.961320



Average of each scaled training feature:
Pregnancies                -0.0
Glucose                    -0.0
BloodPressure               0.0
SkinThickness              -0.0
Insulin                    -0.0
BMI                         0.0
DiabetesPedigreeFunction   -0.0
Age                        -0.0
dtype: float64

Standard deviation of each scaled training feature:
Pregnancies                 1.0
Glucose                     1.0
BloodPressure               1.0
SkinThickness               1.0
Insulin                     1.0
BMI                         1.0
DiabetesPedigreeFunction    1.0
Age                         1.0
dtype: float64


Standardization worked properly:

All training feature means are approximately 0.
All standard deviations are 1.
Training and testing data now use the same learned scaling method.

## 3.5 Feature Selection Using Recursive Feature Elimination

Recursive Feature Elimination ranks the clinical attributes according to their
predictive contribution.

Logistic Regression is used as the RFE estimator. RFE is fitted only on the
training data to prevent information leakage. The five highest-ranked features
are selected initially, and different subset sizes will be evaluated later.

In [11]:
# Create the estimator used for RFE
rfe_estimator = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Select the five highest-ranked features
rfe_selector = RFE(
    estimator=rfe_estimator,
    n_features_to_select=5
)

# Fit RFE only using the scaled training data
rfe_selector.fit(X_train_scaled, y_train)

# Create a feature-ranking table
feature_ranking = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Selected": rfe_selector.support_,
    "RFE_Rank": rfe_selector.ranking_
}).sort_values(
    by=["RFE_Rank", "Feature"]
).reset_index(drop=True)

print("RFE feature ranking:")
display(feature_ranking)

# Store the selected feature names
selected_features = X_train_scaled.columns[
    rfe_selector.support_
].tolist()

print("\nSelected features:")
print(selected_features)

# Create reduced training and testing datasets
X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]

print("\nReduced training data shape:", X_train_selected.shape)
print("Reduced testing data shape:", X_test_selected.shape)

RFE feature ranking:


,Feature,Selected,RFE_Rank
0,Age,True,1
1,BMI,True,1
2,DiabetesPedigreeFunction,True,1
3,Glucose,True,1
4,Pregnancies,True,1
5,Insulin,False,2
6,BloodPressure,False,3
7,SkinThickness,False,4



Selected features:
['Pregnancies', 'Glucose', 'BMI', 'DiabetesPedigreeFunction', 'Age']

Reduced training data shape: (614, 5)
Reduced testing data shape: (154, 5)


RFE selected these five features:

Pregnancies
Glucose
BMI
Diabetes Pedigree Function
Age

The three features removed were:

Insulin
Blood Pressure
Skin Thickness

This also agrees with the earlier correlation analysis. Member 3’s preprocessing and initial feature-selection work is now complete.

## **4.3 Gaussian Naive Bayes**

Gaussian Naive Bayes is a lightweight probabilistic classifier based on Bayes'
theorem. It assumes that the clinical features follow Gaussian distributions
within each outcome class.

The model is computationally efficient and suitable for systems with limited
processing power and memory.

In [13]:
# Create the Gaussian Naive Bayes model
naive_bayes_model = GaussianNB()

# Train using the five selected features
naive_bayes_model.fit(
    X_train_selected,
    y_train
)

# Generate predictions and probabilities
naive_bayes_predictions = naive_bayes_model.predict(
    X_test_selected
)

naive_bayes_probabilities = naive_bayes_model.predict_proba(
    X_test_selected
)[:, 1]

# Calculate training and testing accuracy
naive_bayes_train_accuracy = naive_bayes_model.score(
    X_train_selected,
    y_train
)

naive_bayes_test_accuracy = naive_bayes_model.score(
    X_test_selected,
    y_test
)

print("Gaussian Naive Bayes trained successfully")
print("Training accuracy:", round(naive_bayes_train_accuracy, 4))
print("Testing accuracy:", round(naive_bayes_test_accuracy, 4))

Gaussian Naive Bayes trained successfully
Training accuracy: 0.7834
Testing accuracy: 0.7338


The Gaussian Naive Bayes model trained correctly:

Training accuracy: 78.34%
Testing accuracy: 73.38%
Difference: approximately 4.96%

So far, Gaussian Naive Bayes has the highest testing accuracy and a smaller training-testing gap than the other two lightweight models.